# RealMLP — Pre-tuned MLP for Tabular Data

**Paper**: [Better by Default: Strong Pre-Tuned MLPs and Boosted Trees on Tabular Data](https://openreview.net/forum?id=3BNPUDvqMt) (NeurIPS 2024)  
**Library**: `pytabkit` — `RealMLP_TD_Classifier` (TD = Tuned Defaults, no HPO needed)

Key tricks over a vanilla MLP:
- Piecewise-linear (PLR) embeddings for numerical features
- Careful weight initialization and per-layer learning rates
- Label smoothing, dropout schedule, cosine LR
- One-hot encoding for categoricals (handled internally)

**Baseline to beat**: CatBoost OOF AUC 0.95544 (CPU), LB 0.95357

In [1]:
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from pytabkit import RealMLP_TD_Classifier

BASELINE_AUC = 0.95544  # CPU CatBoost OOF

/home/jon/.local/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /home/jon/.local/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/home/jon/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
KAGGLE_DATA = Path('/kaggle/input/playground-series-s6e2')
LOCAL_DATA  = Path('data')
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(pd.read_csv(DATA_DIR / 'train.csv'))
test  = prep(pd.read_csv(DATA_DIR / 'test.csv'))
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
FEATURES     = NUM_FEATURES + CAT_FEATURES

y = train['heart_disease'].values

# One-hot encode categoricals — RealMLP handles numeric input only
X      = pd.get_dummies(train[FEATURES], columns=CAT_FEATURES).astype(np.float32)
X_test = pd.get_dummies(test[FEATURES],  columns=CAT_FEATURES).astype(np.float32)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print(f'Train: {X.shape}  Test: {X_test.shape}')

Train: (630000, 28)  Test: (270000, 28)


## 5-Fold CV

In [3]:
skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof        = np.zeros(len(y))
test_folds = np.zeros((5, len(X_test)))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'\n=== Fold {fold + 1}/5 ===')

    model = RealMLP_TD_Classifier(
        device       = 'cuda',
        random_state = 42,
        verbosity    = 1,   # show epoch progress
        n_cv         = 1,   # no internal CV — we're doing our own
        n_refit      = 0,
    )
    model.fit(X.iloc[tr_idx].values, y[tr_idx])

    oof[val_idx]     = model.predict_proba(X.iloc[val_idx].values)[:, 1]
    test_folds[fold] = model.predict_proba(X_test.values)[:, 1]

    fold_auc = roc_auc_score(y[val_idx], oof[val_idx])
    print(f'Fold {fold + 1} AUC: {fold_auc:.5f}')

cv_auc     = roc_auc_score(y, oof)
test_preds = test_folds.mean(axis=0)

print(f'\nRealMLP OOF AUC:   {cv_auc:.5f}')
print(f'CatBoost baseline: {BASELINE_AUC:.5f}')
print(f'Delta:             {cv_auc - BASELINE_AUC:+.5f}')


=== Fold 1/5 ===
Columns classified as continuous: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Columns classified as categorical: []


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jon/mambaforge/envs/S4E10/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=256` reached.
GPU available:

Fold 1 AUC: 0.95408

=== Fold 2/5 ===
Columns classified as continuous: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Columns classified as categorical: []


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jon/mambaforge/envs/S4E10/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=256` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jon/mambaforge/envs/

Fold 2 AUC: 0.95310

=== Fold 3/5 ===
Columns classified as continuous: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Columns classified as categorical: []


...

Fold 3 AUC: 0.95373

=== Fold 4/5 ===
Columns classified as continuous: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Columns classified as categorical: []


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jon/mambaforge/envs/S4E10/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=256` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jon/mambaforge/envs/

Fold 4 AUC: 0.95333

=== Fold 5/5 ===
Columns classified as continuous: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Columns classified as categorical: []


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jon/mambaforge/envs/S4E10/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=256` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jon/mambaforge/envs/

Fold 5 AUC: 0.95417

RealMLP OOF AUC:   0.95354
CatBoost baseline: 0.95544
Delta:             -0.00190


## Save OOF + Submit

In [4]:
np.save('submissions/oof_realmlp.npy',  oof)
np.save('submissions/test_realmlp.npy', test_preds)

label = 'realmlp_td_default'
fname = f'submissions/{label}.csv'
sub   = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)

desc = f'{label} | cv_auc={cv_auc:.4f}'
ON_KAGGLE = Path('/kaggle/working').exists()
if ON_KAGGLE:
    print(f'Submit: kaggle competitions submit -c playground-series-s6e2 -f {fname} -m "{desc}"')
else:
    r = subprocess.run(
        ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
         '-f', fname, '-m', desc],
        capture_output=True, text=True
    )
    status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
    print(f'{label}: {status}')
    print(f'desc: {desc}')

realmlp_td_default: 400 Client Error: Bad Request for url: https://www.kaggle.com/api/v1/competitions/submissions/submit/playground-series-s
desc: realmlp_td_default | cv_auc=0.9535


## SLSQP Ensemble with GBTs

In [5]:
from scipy.optimize import minimize

oof_files = {
    'catboost': 'submissions/oof_cat.npy',
    'lgbm':     'submissions/oof_lgb.npy',
    'xgb':      'submissions/oof_xgb.npy',
    'realmlp':  'submissions/oof_realmlp.npy',
}
test_files = {
    'catboost': 'submissions/test_cat.npy',
    'lgbm':     'submissions/test_lgb.npy',
    'xgb':      'submissions/test_xgb.npy',
    'realmlp':  'submissions/test_realmlp.npy',
}

oofs  = {k: np.load(v) for k, v in oof_files.items()}
tests = {k: np.load(v) for k, v in test_files.items()}
names = list(oofs.keys())

print('Individual OOF AUCs:')
for n in names:
    print(f'  {n:<12} {roc_auc_score(y, oofs[n]):.5f}')

oof_list = [oofs[n] for n in names]
res = minimize(
    lambda w: -roc_auc_score(y, sum(wi*o for wi, o in zip(w, oof_list))),
    x0=[1/len(names)] * len(names),
    method='SLSQP',
    bounds=[(0, 1)] * len(names),
    constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1},
)
opt_auc = -res.fun
print(f'\nSLSQP ensemble OOF AUC: {opt_auc:.5f}  (vs CatBoost {BASELINE_AUC:.5f})')
for n, w in zip(names, res.x):
    print(f'  {n:<12} {w:.4f}')

if opt_auc > BASELINE_AUC:
    pred   = sum(w * tests[n] for w, n in zip(res.x, names))
    fname2 = 'submissions/ensemble_gbt_realmlp.csv'
    desc2  = f'ensemble_gbt_realmlp | cv_auc={opt_auc:.4f}'
    ss.copy().assign(**{'Heart Disease': pred}).to_csv(fname2, index=False)
    if not ON_KAGGLE:
        r2 = subprocess.run(
            ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
             '-f', fname2, '-m', desc2],
            capture_output=True, text=True
        )
        print(f'ensemble: {"submitted" if "successfully" in r2.stdout.lower() else r2.stdout[:80]}')
else:
    print('RealMLP did not improve ensemble — skipping.')

Individual OOF AUCs:
  catboost     0.95532
  lgbm         0.95523
  xgb          0.95525
  realmlp      0.95354

SLSQP ensemble OOF AUC: 0.95522  (vs CatBoost 0.95544)
  catboost     0.2500
  lgbm         0.2500
  xgb          0.2500
  realmlp      0.2500
RealMLP did not improve ensemble — skipping.
